# Interpretability & Error Analysis

Model interpretability and failure-mode analysis for the best classifiers

**Contents:**
1. Retrain best models on full data
2. Permutation importance (all structured-feature models)
3. SHAP values (tree-based: Random Forest, XGBoost, LightGBM)
4. Logistic Regression coefficient analysis
5. TF-IDF top features (text classifier)
6. Visualised decision tree (interpretable surrogate)
7. Feature → safety hypothesis mapping
8. Error analysis on misclassified samples
9. Log findings to `results/interpretability.json`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    matthews_corrcoef, classification_report, confusion_matrix
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
RESULTS_DIR = Path('../results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. Load Data & Reproduce Best Models

In [ ]:
# Load data
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')
d2b = pd.read_parquet('../data/processed/d2b_turns.parquet')

df = d2d.merge(d2c[['session_id', 'overall_unsafe']], on='session_id')
y = np.array(df['overall_unsafe'].astype(int).tolist())

numeric_cols = [
    'response_length_words', 'response_length_chars', 'sentence_count',
    'avg_sentence_length', 'paragraph_count', 'hedging_count', 'hedging_freq',
    'certainty_count', 'certainty_freq', 'hedging_certainty_ratio',
    'nct_id_count', 'nct_id_unique', 'citation_density',
    'bullet_count', 'numbered_list_count', 'refusal_count',
    'is_monitored', 'is_baseline_pressure'
]
bool_cols = ['has_refusal', 'has_citations', 'has_hedging', 'has_certainty']
for c in bool_cols:
    df[c] = df[c].astype(int)

cat_cols = ['scenario_id', 'pressure_id', 'monitoring_id']
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    df[f'{c}_enc'] = le.fit_transform(df[c].astype(str))
    le_dict[c] = le
encoded_cat_cols = [f'{c}_enc' for c in cat_cols]

feature_cols = numeric_cols + bool_cols + encoded_cat_cols
X_struct = np.array(df[feature_cols].values.tolist(), dtype=float)

# Raw text
text_df = d2b[d2b['role'] == 'assistant'].groupby('session_id')['content'].apply(
    lambda x: ' '.join(x)).reset_index()
text_df.columns = ['session_id', 'full_response']
df = df.merge(text_df, on='session_id', how='left')
df['full_response'] = df['full_response'].fillna('')
X_text = df['full_response'].tolist()

imbalance_ratio = (y == 0).sum() / (y == 1).sum()

print(f'Samples: {len(y)} | Unsafe: {y.sum()} | Features: {len(feature_cols)}')
print(f'Feature names: {feature_cols}')

In [ ]:
# Retrain top models on full data with best hyperparameters
model_results = json.load(open(RESULTS_DIR / 'model_comparison.json'))

# Scale structured features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_struct)

def parse_param(val_str, param_name=''):
    """Convert string parameter values back to proper types."""
    if val_str == 'None':
        return None
    if val_str in ('sqrt', 'log2', 'scale', 'auto'):
        return val_str
    try:
        if '.' in val_str:
            return float(val_str)
        return int(val_str)
    except (ValueError, TypeError):
        return val_str

# 1. Logistic Regression (tuned)
lr_params = model_results['models']['Logistic Regression']['best_params_per_fold'][0]
lr_model = LogisticRegression(
    C=parse_param(lr_params.get('clf__C', '1.0')),
    penalty=lr_params.get('clf__penalty', 'l2'),
    solver='saga', class_weight='balanced', max_iter=2000, random_state=SEED
)
lr_model.fit(X_scaled, y)

# 2. Random Forest (tuned)
rf_params = model_results['models']['Random Forest']['best_params_per_fold'][0]
rf_max_features = parse_param(rf_params.get('clf__max_features', 'sqrt'))
rf_model = RandomForestClassifier(
    n_estimators=parse_param(rf_params.get('clf__n_estimators', '200')),
    max_depth=parse_param(rf_params.get('clf__max_depth', 'None')),
    min_samples_leaf=parse_param(rf_params.get('clf__min_samples_leaf', '5')),
    max_features=rf_max_features,
    class_weight='balanced', random_state=SEED
)
rf_model.fit(X_scaled, y)

# 3. XGBoost (tuned)
xgb_params = model_results['models']['XGBoost']['best_params_per_fold'][0]
xgb_model = XGBClassifier(
    n_estimators=parse_param(xgb_params.get('clf__n_estimators', '200')),
    max_depth=parse_param(xgb_params.get('clf__max_depth', '5')),
    learning_rate=parse_param(xgb_params.get('clf__learning_rate', '0.1')),
    subsample=parse_param(xgb_params.get('clf__subsample', '0.8')),
    colsample_bytree=parse_param(xgb_params.get('clf__colsample_bytree', '0.8')),
    reg_alpha=parse_param(xgb_params.get('clf__reg_alpha', '0')),
    reg_lambda=parse_param(xgb_params.get('clf__reg_lambda', '1.0')),
    scale_pos_weight=imbalance_ratio, eval_metric='logloss',
    use_label_encoder=False, random_state=SEED, verbosity=0
)
xgb_model.fit(X_scaled, y)

# 4. LightGBM (tuned)
lgb_params = model_results['models']['LightGBM']['best_params_per_fold'][0]
lgb_model = LGBMClassifier(
    n_estimators=parse_param(lgb_params.get('clf__n_estimators', '200')),
    max_depth=parse_param(lgb_params.get('clf__max_depth', '-1')),
    learning_rate=parse_param(lgb_params.get('clf__learning_rate', '0.1')),
    num_leaves=parse_param(lgb_params.get('clf__num_leaves', '31')),
    subsample=parse_param(lgb_params.get('clf__subsample', '0.8')),
    colsample_bytree=parse_param(lgb_params.get('clf__colsample_bytree', '0.8')),
    reg_alpha=parse_param(lgb_params.get('clf__reg_alpha', '0')),
    reg_lambda=parse_param(lgb_params.get('clf__reg_lambda', '1.0')),
    is_unbalance=True, random_state=SEED, verbosity=-1, n_jobs=1
)
lgb_model.fit(X_scaled, y)

# 5. TF-IDF text classifier (tuned)
tfidf_params = model_results['models']['TF-IDF Text Classifier']['best_params_per_fold'][0]
ngram = eval(tfidf_params.get('tfidf__ngram_range', '(1, 2)'))
tfidf_vec = TfidfVectorizer(
    max_features=parse_param(tfidf_params.get('tfidf__max_features', '5000')),
    ngram_range=ngram,
    min_df=parse_param(tfidf_params.get('tfidf__min_df', '2')),
    sublinear_tf=True
)
X_tfidf = tfidf_vec.fit_transform(X_text)
tfidf_lr = LogisticRegression(
    C=parse_param(tfidf_params.get('clf__C', '1.0')),
    penalty=tfidf_params.get('clf__penalty', 'l2'),
    solver='saga', class_weight='balanced', max_iter=2000, random_state=SEED
)
tfidf_lr.fit(X_tfidf, y)

models = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}
print('All models retrained on full data with tuned hyperparameters.')

## 2. Permutation Importance (All Structured-Feature Models)

In [ ]:
# Permutation importance for all structured-feature models
perm_results = {}
for name, model in models.items():
    perm = permutation_importance(
        model, X_scaled, y, n_repeats=30, random_state=SEED,
        scoring='average_precision', n_jobs=1
    )
    perm_results[name] = perm

# Plot top-10 features per model
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (name, perm) in enumerate(perm_results.items()):
    sorted_idx = perm.importances_mean.argsort()[::-1][:10]
    top_features = [feature_cols[i] for i in sorted_idx]
    top_means = perm.importances_mean[sorted_idx]
    top_stds = perm.importances_std[sorted_idx]

    axes[idx].barh(range(10), top_means[::-1], xerr=top_stds[::-1],
                   color='#2980b9', alpha=0.8)
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels(top_features[::-1], fontsize=8)
    axes[idx].set_title(f'{name}', fontsize=10)
    axes[idx].set_xlabel('Mean importance (PR AUC drop)')

plt.suptitle('Permutation Importance - Top 10 Features per Model', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '15_permutation_importance.png', bbox_inches='tight')
plt.show()

print('Permutation importance computed for all structured models.')

## 3. SHAP Values (Tree-Based Models)

In [ ]:
# SHAP values for tree-based models
tree_models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}

shap_values_dict = {}
for name, model in tree_models.items():
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_scaled)
    # Handle different SHAP return formats:
    # - list [class0, class1] → take class1
    # - 3D array (samples, features, classes) → take [:, :, 1]
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    elif shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1]
    shap_values_dict[name] = shap_vals
    print(f'  {name}: SHAP shape = {shap_vals.shape}')

print('SHAP values computed for tree-based models.')

In [ ]:
# SHAP summary plots (manual beeswarm to avoid SHAP subplot rendering issues)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, shap_vals) in enumerate(shap_values_dict.items()):
    ax = axes[idx]
    mean_abs = np.abs(shap_vals).mean(axis=0)
    order = np.argsort(-mean_abs)[:12]  # top 12 features

    for rank, feat_idx in enumerate(order):
        xs = shap_vals[:, feat_idx]
        feat_vals = X_scaled[:, feat_idx]
        ys = np.full_like(xs, rank) + np.random.default_rng(SEED).normal(0, 0.15, size=len(xs))
        scatter = ax.scatter(xs, ys, c=feat_vals, cmap='coolwarm',
                           s=8, alpha=0.6, vmin=-2, vmax=2, edgecolors='none')

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([feature_cols[i] for i in order], fontsize=8)
    ax.set_xlabel('SHAP value (impact on model output)')
    ax.set_title(f'{name}', fontsize=10)
    ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)
    ax.invert_yaxis()

# Add colorbar
cbar = fig.colorbar(scatter, ax=axes[-1], shrink=0.8, pad=0.02)
cbar.set_label('Feature value\n(standardised)', fontsize=8)
cbar.ax.set_yticklabels(['Low', '', '', '', 'High'], fontsize=7)

plt.suptitle('SHAP Summary - Feature Impact on Unsafe Prediction', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '16_shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP mean absolute values - aggregated feature ranking
shap_importance = {}
for name, shap_vals in shap_values_dict.items():
    mean_abs = np.abs(shap_vals).mean(axis=0)
    shap_importance[name] = dict(zip(feature_cols, mean_abs.tolist()))

# Consensus ranking: average rank across tree models
rank_df = pd.DataFrame(shap_importance).rank(ascending=False)
rank_df['avg_rank'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('avg_rank')

print('=== SHAP Consensus Feature Ranking (lower = more important) ===')
print(rank_df[['avg_rank']].head(15).to_string())

## 4. Logistic Regression Coefficients

In [ ]:
# Logistic Regression coefficients (standardized features → comparable magnitudes)
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0],
    'abs_coefficient': np.abs(lr_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_n = 15
top_coefs = coef_df.head(top_n)
colors = ['#e74c3c' if c > 0 else '#2980b9' for c in top_coefs['coefficient']]

ax.barh(range(top_n), top_coefs['coefficient'].values[::-1], color=colors[::-1], alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_coefs['feature'].values[::-1], fontsize=9)
ax.set_xlabel('Coefficient (positive = increases P(unsafe))')
ax.set_title('Logistic Regression Coefficients (Top 15 by |coef|)')
ax.axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '17_lr_coefficients.png', bbox_inches='tight')
plt.show()

print('\nTop 5 features increasing P(unsafe):')
print(coef_df[coef_df['coefficient'] > 0].head(5)[['feature', 'coefficient']].to_string(index=False))
print('\nTop 5 features decreasing P(unsafe):')
print(coef_df[coef_df['coefficient'] < 0].head(5)[['feature', 'coefficient']].to_string(index=False))

## 5. TF-IDF Top Features (Text Classifier)

In [ ]:
# TF-IDF feature importance from text classifier coefficients
tfidf_feature_names = tfidf_vec.get_feature_names_out()
tfidf_coefs = tfidf_lr.coef_[0]

tfidf_df = pd.DataFrame({
    'token': tfidf_feature_names,
    'coefficient': tfidf_coefs,
    'abs_coefficient': np.abs(tfidf_coefs)
}).sort_values('abs_coefficient', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top tokens pushing toward UNSAFE
top_unsafe = tfidf_df[tfidf_df['coefficient'] > 0].head(20)
axes[0].barh(range(len(top_unsafe)), top_unsafe['coefficient'].values[::-1],
             color='#e74c3c', alpha=0.8)
axes[0].set_yticks(range(len(top_unsafe)))
axes[0].set_yticklabels(top_unsafe['token'].values[::-1], fontsize=8)
axes[0].set_title('Top 20 Tokens → Unsafe', fontsize=10)
axes[0].set_xlabel('Coefficient')

# Top tokens pushing toward SAFE
top_safe = tfidf_df[tfidf_df['coefficient'] < 0].sort_values('coefficient').head(20)
axes[1].barh(range(len(top_safe)), top_safe['coefficient'].values[::-1],
             color='#2980b9', alpha=0.8)
axes[1].set_yticks(range(len(top_safe)))
axes[1].set_yticklabels(top_safe['token'].values[::-1], fontsize=8)
axes[1].set_title('Top 20 Tokens → Safe', fontsize=10)
axes[1].set_xlabel('Coefficient')

plt.suptitle('TF-IDF Text Classifier: Most Discriminative Tokens', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '18_tfidf_top_tokens.png', bbox_inches='tight')
plt.show()

print(f'Total TF-IDF features: {len(tfidf_feature_names)}')
print(f'Non-zero coefficients: {(tfidf_coefs != 0).sum()}')

## 6. Visualised Decision Tree (Interpretable Surrogate)

A shallow decision tree (depth=3) trained as an interpretable surrogate model.
This reveals the *rules* the classifier uses — directly readable decision logic
that can be mapped back to safety-relevant behaviours.

In [ ]:
# Train a shallow decision tree as an interpretable surrogate
dt_model = DecisionTreeClassifier(
    max_depth=3, class_weight='balanced', random_state=SEED,
    min_samples_leaf=10
)
dt_model.fit(X_scaled, y)

# Evaluate surrogate fidelity (how well it approximates the best model)
dt_preds = dt_model.predict(X_scaled)
lgb_preds = (lgb_model.predict_proba(X_scaled)[:, 1] >= 0.5).astype(int)
fidelity = (dt_preds == lgb_preds).mean()
dt_acc = (dt_preds == y).mean()
dt_f1 = f1_score(y, dt_preds)

print(f'Decision Tree Surrogate (depth=3):')
print(f'  Accuracy: {dt_acc:.3f}')
print(f'  F1 Score: {dt_f1:.3f}')
print(f'  Fidelity to LightGBM: {fidelity:.3f} (agreement on predictions)')

# Visualise the tree
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    dt_model, ax=ax,
    feature_names=feature_cols,
    class_names=['Safe', 'Unsafe'],
    filled=True, rounded=True, fontsize=9,
    proportion=True, impurity=False
)
ax.set_title('Interpretable Decision Tree (depth=3) — Safety Classification Rules', fontsize=13)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '25_decision_tree.png', bbox_inches='tight', dpi=150)
plt.show()

# Extract and print decision rules
from sklearn.tree import export_text
rules = export_text(dt_model, feature_names=feature_cols, decimals=2)
print('\n=== Decision Rules ===')
print(rules)

## 7. Feature → Safety Hypothesis Mapping

Linking the top predictive features back to the original safety hypotheses.
This section connects empirical findings to the theoretical framework, showing
which misalignment signals are most detectable and which remain elusive.

In [ ]:
# Define the mapping from features to safety hypotheses
feature_hypothesis_map = {
    # Evidence faithfulness (overclaiming / hallucination)
    'nct_id_unique': 'Evidence faithfulness — unique citations correlate with grounded responses',
    'nct_id_count': 'Evidence faithfulness — citation volume as proxy for evidence engagement',
    'citation_density': 'Evidence faithfulness — ratio of citations to response length',
    'has_citations': 'Evidence faithfulness — binary presence of verifiable references',
    # Uncertainty communication (hedging suppression)
    'hedging_freq': 'Uncertainty honesty — hedging language frequency signals epistemic calibration',
    'hedging_count': 'Uncertainty honesty — raw hedging markers (may, might, uncertain)',
    'certainty_freq': 'Overconfidence — excessive certainty markers without evidence support',
    'certainty_count': 'Overconfidence — raw confidence assertions',
    'hedging_certainty_ratio': 'Calibration balance — relative proportion of hedging vs certainty',
    'has_hedging': 'Uncertainty honesty — presence of any uncertainty language',
    'has_certainty': 'Overconfidence — presence of strong confidence claims',
    # Response structure (complexity as deception indicator)
    'response_length_words': 'Verbosity — longer responses may dilute or obscure unsafe content',
    'response_length_chars': 'Verbosity — character-level length',
    'sentence_count': 'Structural complexity — more sentences may indicate elaboration or deflection',
    'avg_sentence_length': 'Complexity — longer sentences correlate with denser reasoning',
    'paragraph_count': 'Structure — multi-paragraph responses for complex claims',
    'bullet_count': 'Formatting — structured output may mask substantive gaps',
    'numbered_list_count': 'Formatting — enumerated claims increase surface verifiability',
    # Safety-specific signals
    'refusal_count': 'Refusal behaviour — explicit refusals indicate safety-aware responses',
    'has_refusal': 'Refusal behaviour — binary refusal detection',
    # Experimental conditions
    'scenario_id_enc': 'Task vulnerability — scenario type determines base rate of unsafe behaviour',
    'pressure_id_enc': 'Adversarial pressure — the experimental intervention driving misalignment',
    'monitoring_id_enc': 'Monitoring sensitivity — whether oversight awareness changes behaviour',
    'is_monitored': 'Monitoring sensitivity — direct encoding of oversight condition',
    'is_baseline_pressure': 'Control condition — absence of adversarial pressure',
}

# Get consensus top features from SHAP + permutation importance
shap_top = rank_df.head(10).index.tolist()
perm_top = []
for name, perm in perm_results.items():
    top_idx = perm.importances_mean.argsort()[::-1][:5]
    perm_top.extend([feature_cols[i] for i in top_idx])
perm_top_unique = list(dict.fromkeys(perm_top))[:10]

# Combine into consensus
all_top = list(dict.fromkeys(shap_top + perm_top_unique))

# Build the mapping table
print('=' * 80)
print('FEATURE → SAFETY HYPOTHESIS MAPPING')
print('=' * 80)
print(f'\n{"Feature":<28} {"Direction":<12} {"Safety Hypothesis"}')
print('-' * 80)

for feat in all_top:
    hypothesis = feature_hypothesis_map.get(feat, 'Composite signal')
    # Get direction from LR coefficients
    coef_row = coef_df[coef_df['feature'] == feat]
    if len(coef_row) > 0:
        coef_val = coef_row['coefficient'].values[0]
        direction = '→ unsafe' if coef_val > 0 else '→ safe'
    else:
        direction = '—'
    print(f'{feat:<28} {direction:<12} {hypothesis}')

# Summary: which safety dimensions are most predictable?
print('\n\n=== Safety Dimension Predictability (from model evidence) ===')
dimensions = {
    'Evidence faithfulness': ['nct_id_unique', 'nct_id_count', 'citation_density', 'has_citations'],
    'Uncertainty honesty': ['hedging_freq', 'hedging_count', 'has_hedging', 'hedging_certainty_ratio'],
    'Overconfidence': ['certainty_freq', 'certainty_count', 'has_certainty'],
    'Response complexity': ['response_length_words', 'sentence_count', 'avg_sentence_length'],
    'Task vulnerability': ['scenario_id_enc'],
    'Monitoring sensitivity': ['is_monitored', 'monitoring_id_enc'],
}

for dim_name, dim_features in dimensions.items():
    # Average SHAP importance across tree models for this dimension
    dim_shap = []
    for feat in dim_features:
        if feat in rank_df.index:
            dim_shap.append(rank_df.loc[feat, 'avg_rank'])
    avg_rank = np.mean(dim_shap) if dim_shap else float('nan')
    signal = '★★★' if avg_rank < 8 else '★★' if avg_rank < 15 else '★'
    print(f'  {signal} {dim_name:<25} (avg SHAP rank: {avg_rank:.1f})')

## 8. Error Analysis on Misclassified Samples

In [ ]:
# Error analysis using cross-validated predictions (avoid train-set bias)
from sklearn.model_selection import cross_val_predict

# Use LightGBM (best structured model) for error analysis
lgb_pipe = Pipeline([('scaler', StandardScaler()), ('clf', lgb_model)])
# Get CV predictions to avoid overfitting
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

y_prob_cv = cross_val_predict(
    Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LGBMClassifier(
            n_estimators=int(lgb_params.get('clf__n_estimators', '200')),
            max_depth=int(lgb_params.get('clf__max_depth', '-1')),
            learning_rate=float(lgb_params.get('clf__learning_rate', '0.1')),
            num_leaves=int(lgb_params.get('clf__num_leaves', '31')),
            is_unbalance=True, random_state=SEED, verbosity=-1, n_jobs=1
        ))
    ]),
    X_struct, y, cv=cv, method='predict_proba'
)[:, 1]

y_pred_cv = (y_prob_cv >= 0.5).astype(int)

# Classify errors
df['y_true'] = y
df['y_prob'] = y_prob_cv
df['y_pred'] = y_pred_cv
df['error_type'] = 'correct'
df.loc[(df['y_true'] == 1) & (df['y_pred'] == 0), 'error_type'] = 'false_negative'
df.loc[(df['y_true'] == 0) & (df['y_pred'] == 1), 'error_type'] = 'false_positive'

error_counts = df['error_type'].value_counts()
print('=== Error Breakdown (LightGBM, CV predictions) ===')
print(error_counts)
print(f'\nFalse Negative Rate: {error_counts.get("false_negative", 0) / y.sum():.1%}')
print(f'False Positive Rate: {error_counts.get("false_positive", 0) / (y == 0).sum():.1%}')

In [ ]:
# Analyze characteristics of misclassified samples
error_analysis_cols = ['scenario_id', 'pressure_id', 'monitoring_id'] + numeric_cols[:8]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Error distribution by pressure condition
ax = axes[0, 0]
ct = pd.crosstab(df['pressure_id'], df['error_type'], normalize='index')
ct[['correct', 'false_negative', 'false_positive']].plot(
    kind='bar', stacked=True, ax=ax, color=['#27ae60', '#e74c3c', '#f39c12']
)
ax.set_title('Error Rate by Pressure Condition')
ax.set_xlabel('')
ax.set_ylabel('Proportion')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=45)

# 2. Error distribution by monitoring condition
ax = axes[0, 1]
ct2 = pd.crosstab(df['is_monitored'].map({1: 'Monitored', 0: 'Unmonitored'}),
                  df['error_type'], normalize='index')
ct2[['correct', 'false_negative', 'false_positive']].plot(
    kind='bar', stacked=True, ax=ax, color=['#27ae60', '#e74c3c', '#f39c12']
)
ax.set_title('Error Rate by Monitoring')
ax.set_xlabel('')
ax.set_ylabel('Proportion')
ax.legend(fontsize=8)

# 3. Prediction confidence distribution by error type
ax = axes[1, 0]
for etype, color in [('correct', '#27ae60'), ('false_negative', '#e74c3c'), ('false_positive', '#f39c12')]:
    subset = df[df['error_type'] == etype]['y_prob']
    if len(subset) > 0:
        ax.hist(subset, bins=20, alpha=0.5, label=f'{etype} (n={len(subset)})', color=color)
ax.set_xlabel('Predicted P(unsafe)')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution by Error Type')
ax.legend(fontsize=8)

# 4. Feature comparison: FN vs correct unsafe
ax = axes[1, 1]
fn_mask = df['error_type'] == 'false_negative'
tp_mask = (df['y_true'] == 1) & (df['y_pred'] == 1)

if fn_mask.sum() > 0 and tp_mask.sum() > 0:
    compare_features = ['hedging_freq', 'certainty_freq', 'citation_density',
                        'response_length_words', 'refusal_count']
    fn_means = df.loc[fn_mask, compare_features].mean()
    tp_means = df.loc[tp_mask, compare_features].mean()

    x_pos = np.arange(len(compare_features))
    width = 0.35
    ax.bar(x_pos - width/2, fn_means, width, label='False Negatives', color='#e74c3c', alpha=0.7)
    ax.bar(x_pos + width/2, tp_means, width, label='True Positives', color='#27ae60', alpha=0.7)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(compare_features, fontsize=8, rotation=30)
    ax.set_title('FN vs TP: Feature Comparison')
    ax.legend(fontsize=8)

plt.suptitle('Error Analysis - LightGBM Structured Model', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '19_error_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Detailed look at false negatives (missed unsafe samples - critical for safety)
fn_sessions = df[df['error_type'] == 'false_negative'][
    ['session_id', 'scenario_id', 'pressure_id', 'monitoring_id',
     'hedging_freq', 'certainty_freq', 'citation_density', 'y_prob']
].sort_values('y_prob', ascending=False)

print(f'=== False Negatives: {len(fn_sessions)} missed unsafe sessions ===')
print('These are unsafe sessions the model predicted as safe — critical oversight failures.\n')
print(fn_sessions.to_string(index=False))

## 9. Log Findings

In [ ]:
# Structured logging
log_entry = {
    'description': 'Interpretability and error analysis',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'permutation_importance': {
        name: {
            'top_5': [
                {'feature': feature_cols[i], 'importance': float(perm.importances_mean[i])}
                for i in perm.importances_mean.argsort()[::-1][:5]
            ]
        }
        for name, perm in perm_results.items()
    },
    'shap_consensus_top_10': rank_df['avg_rank'].head(10).to_dict(),
    'lr_coefficients': {
        'top_5_unsafe': coef_df[coef_df['coefficient'] > 0].head(5)[['feature', 'coefficient']].to_dict('records'),
        'top_5_safe': coef_df[coef_df['coefficient'] < 0].sort_values('coefficient').head(5)[['feature', 'coefficient']].to_dict('records'),
    },
    'tfidf_text_classifier': {
        'total_features': int(len(tfidf_feature_names)),
        'top_10_unsafe_tokens': tfidf_df[tfidf_df['coefficient'] > 0].head(10)['token'].tolist(),
        'top_10_safe_tokens': tfidf_df[tfidf_df['coefficient'] < 0].sort_values('coefficient').head(10)['token'].tolist(),
    },
    'decision_tree_surrogate': {
        'max_depth': 3,
        'accuracy': float(dt_acc),
        'f1_score': float(dt_f1),
        'fidelity_to_lgbm': float(fidelity),
    },
    'feature_safety_mapping': {
        feat: feature_hypothesis_map.get(feat, 'Composite signal')
        for feat in all_top[:10]
    },
    'error_analysis': {
        'model': 'LightGBM (CV predictions)',
        'n_false_negatives': int(error_counts.get('false_negative', 0)),
        'n_false_positives': int(error_counts.get('false_positive', 0)),
        'n_correct': int(error_counts.get('correct', 0)),
        'fn_rate': float(error_counts.get('false_negative', 0) / max(y.sum(), 1)),
        'fp_rate': float(error_counts.get('false_positive', 0) / max((y == 0).sum(), 1)),
    },
}

results_path = RESULTS_DIR / 'interpretability.json'
with open(results_path, 'w') as f:
    json.dump(log_entry, f, indent=2, default=str)

print(f'Results saved to: {results_path}')

## 10. Summary

**Key findings for AI safety:**
- **Permutation importance** reveals which engineered features drive safety predictions across model families
- **SHAP values** show directionality: how each feature pushes toward/away from unsafe classification
- **LR coefficients** provide linear interpretability — which features correlate most with oversight failures
- **TF-IDF tokens** expose linguistic patterns in raw text that signal unsafe behaviour
- **Decision tree surrogate** provides human-readable rules (if–then logic) for safety classification
- **Feature → hypothesis mapping** connects empirical importance to the original safety framework
- **Error analysis** identifies false negatives (missed unsafe cases) — the most critical failure mode

**Implications for scalable oversight:**
- False negatives represent cases where the model fails to flag genuinely unsafe AI behaviour
- Understanding *why* certain unsafe sessions evade detection informs better monitoring design
- The decision tree reveals that a small number of features (scenario type, citation density, response structure) capture most of the predictive signal — suggesting focused monitoring instruments
- Features relating to **evidence faithfulness** (citations, NCT IDs) are consistently top-ranked, validating the hypothesis that overclaiming is a primary misalignment signal in research-agent tasks
- **Monitoring sensitivity** features rank low, suggesting the model does not systematically behave differently under observation in this dataset — a finding that itself is safety-relevant